# 03 · Download climate data and build daily series per zone

Downloads MeteoSwiss open data, clips it to the AOI and turns it into daily series for every topographic zone of `01`:

| Source | Use | Variables |
|---|---|---|
| Gridded observations (1 km, `ogd-surface-derived-grid`) | training | mean, minimum and maximum air temperature, precipitation, relative sunshine duration |
| CH2025 DAILY-GRIDDED (1 km) | projection | mean, minimum and maximum air temperature, precipitation |

- **Zone values:** a zone's value is the mean over the 1 km cells it covers, weighted by the share of the zone's pixels in each cell.
- **Temperatures** are shifted to the mean elevation of the zone pixels with monthly lapse rates, fitted from the gridded observations around the AOI.

### CH2025 climate scenarios

CH2025 scenarios are **not forecasts of specific years**. Each file holds 30 years of daily weather (model years 1–30, in a 365-day calendar without leap days) that is plausible for one climate state. In the code these states are called *periods*:

| Period | Climate state |
|---|---|
| `ref91-20` | reference climate 1991–2020, the period the scenarios are bias-adjusted to |
| `gwl1.5` | global warming level (GWL) +1.5 °C: global mean temperature 1.5 °C above pre-industrial |
| `gwl2.0` | GWL +2.0 °C |
| `gwl2.5` | GWL +2.5 °C |
| `gwl3.0` | GWL +3.0 °C (missing for `clmcom-cclm4-mpiesm` and `mpicsc-remo1-mpiesm`) |

A GWL is a level of warming, not a date: every model chain reaches it in a different year.

`CH2025_GRID_MODELS` in `config.py` restricts the grid download to some of these chains.

Citation: MeteoSwiss & ETH Zurich (2025): Climate CH2025 – Daily Datasets.

- **Settings:** `config.py` (years, gridded products, CH2025 variables / models)
- **Reads:** `cache/<AOI>_zones/zones.tif` (from `01`)
- **Writes** to `cache/climate/sources/` (shared by all AOIs): the clipped source files
- **Writes** to `cache/climate/<AOI>/`:
  - `obs_grid_zones.nc`, `ch2025_grid_zones.nc`
  - `lapse_rates.csv`, `zone_cells.csv`

Finished years and scenario files are not downloaded again, also not for another AOI as long as the cached files cover its box (AOI cells + `GRID_BUFFER`). If they do not, they are downloaded once more and clipped to the box that holds both AOIs. The current year is refreshed on every run. The CH2025 grid is large: 512 files of ~1.8 GB, each downloaded, clipped and deleted, which takes several hours the first time. An interrupted run resumes where it stopped.

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import (add_calendar, clip_grid, download, fetch_bbox, fit_lapse_rates, grid_bbox, grid_extent,
                             grid_to_zones, stac_get, stac_items, zone_cells, STAC_URL)

DOWNLOAD_CLIMATE = True      # False: only rebuild the zone series from the cached files
DOWNLOAD_CH2025_GRID = True  # the large CH2025 grid download (hours on the first run)
GRID_WORKERS = 3             # parallel CH2025 grid downloads

OBS_COLLECTION = "ch.meteoschweiz.ogd-surface-derived-grid"
GRID_COLLECTION = "ch.meteoschweiz.ogd-climate-scenarios-ch2025-grid"
TMP_DIR = CLIMATE_SOURCE_DIR / "tmp"
ENCODING = {"zlib": True, "complevel": 4, "dtype": "float32"}

this_year = pd.Timestamp.today().year
obs_years = list(range(FIRST_YEAR - (1 if CLIMATE_SPINUP_DAYS else 0), min(LAST_YEAR, this_year) + 1))
obs_start = pd.Timestamp(START) - pd.Timedelta(days=CLIMATE_SPINUP_DAYS)
CLIMATE_DIR.mkdir(parents=True, exist_ok=True)
print(f"AOI: {AOI_NAME} | observations {obs_start.date()} - {min(LAST_YEAR, this_year)}\n"
      f"zone series: {CLIMATE_DIR} | shared source files: {CLIMATE_SOURCE_DIR}")


def save(ds, path):
    ds.to_netcdf(path, encoding={v: ENCODING for v in ds.data_vars})


def load_clipped(path):
    # A shared source file cut to this AOI's box
    with xr.open_dataset(path) as ds:
        return clip_grid(ds, bbox).load()

## 1. Zones on the 1 km climate grid

Every 10 m zone pixel is assigned to its 1 km LV95 cell.

- `zone_cells.csv` holds, per zone and cell, the pixel share (`weight`) and the mean elevation of the zone pixels (`zone_elev`).
- `cell_elev` is the mean elevation of the whole cell, used to fit and apply the lapse rate.

In [ ]:
weights, cells = zone_cells()
bbox = grid_bbox(weights)
weights.to_csv(CLIMATE_DIR / "zone_cells.csv", index=False)
summary = weights.groupby("zone").agg(cells=("E", "size"), pixels=("n", "sum"))
summary["elev_offset_m"] = (weights.weight * (weights.zone_elev - weights.cell_elev)).groupby(weights.zone).sum()
display(summary.round(1))

fig, ax = plt.subplots(figsize=(6, 5))
w = weights[weights.zone == "all"]
sc = ax.scatter(w.E, w.N, c=w.n, s=120, marker="s", cmap="Greens")
plt.colorbar(sc, label="AOI pixels per 1 km cell")
ax.set_aspect("equal")
ax.set_title("1 km climate cells covering the AOI")
plt.show()

## 2. Gridded observations

For each variable and year, the best available version is used:
1. the final yearly archive;
2. otherwise the consolidated monthly files;
3. otherwise the preliminary daily files of the last weeks (precipitation: `RprelimD`).

In [ ]:
PRODUCT_VAR = {product: var for var, product in OBS_GRID_PRODUCTS.items()}
PRODUCT_VAR["rprelimd"] = "pr"
SOURCE_RE = re.compile(r"\.([a-z]+)_ch0\w+?\.swiss\.lv95_(\d{8})\d{6}_\d{14}\.nc$")
PRIORITY = {"archive": 0, "monthly": 1, "daily": 2}


def obs_sources():
    rows = []
    for item in stac_items(OBS_COLLECTION):
        for name, asset in item["assets"].items():
            m = SOURCE_RE.search(name)
            if not m or m.group(1) not in PRODUCT_VAR:
                continue
            kind = "archive" if "-archive." in name else "monthly" if "-last." in name else "daily"
            rows.append({"var": PRODUCT_VAR[m.group(1)], "year": int(m.group(2)[:4]), "kind": kind,
                         "priority": PRIORITY[kind], "name": name, "href": asset["href"]})
    return pd.DataFrame(rows)


def obs_year(var, year, sources, box):
    # One variable and year, clipped to box; archive only if available, else monthly + daily files
    src = sources[(sources["var"] == var) & (sources.year == year)].sort_values(["priority", "name"])
    if (src.kind == "archive").any():
        src = src[src.kind == "archive"]
    parts = []
    for row in src.itertuples():
        path = download(row.href, TMP_DIR / row.name)
        with xr.open_dataset(path) as ds:
            name = [v for v in ds.data_vars if v != "swiss_lv95_coordinates"][0]
            parts.append(clip_grid(ds[[name]].rename({name: var}), box).load())
        path.unlink()
    out = xr.concat(parts, "time")
    out = out.isel(time=np.argsort(out["time"].values, kind="stable"))   # stable: higher priority first per day
    out = out.isel(time=~out.indexes["time"].duplicated())
    out.attrs["complete"] = int((src.kind == "archive").all())
    return out


obs_dir = CLIMATE_SOURCE_DIR / "obs_grid"
obs_dir.mkdir(parents=True, exist_ok=True)
if DOWNLOAD_CLIMATE:
    sources = obs_sources()
    for var in OBS_GRID_PRODUCTS:
        for year in obs_years:
            path = obs_dir / f"{var}_{year}.nc"
            box = fetch_bbox(path, bbox)
            if box is None:   # cached file covers this AOI
                with xr.open_dataset(path) as cached:
                    if cached.attrs.get("complete"):
                        continue
                    box = grid_extent(cached)   # recent year: refreshed with the cached extent
            try:
                ds = obs_year(var, year, sources, box)
                save(ds, path)
                print(f"{var} {year}: {ds.sizes['time']} days ({'final' if ds.attrs['complete'] else 'recent'})")
            except Exception as e:   # e.g. connection lost; retried on the next run
                print(f"{var} {year}: failed {e!r}")

obs_grid = xr.merge([xr.concat([load_clipped(p) for p in sorted(obs_dir.glob(f"{var}_*.nc"))], "time")
                     for var in OBS_GRID_PRODUCTS], join="outer", compat="override")
obs_grid = add_calendar(obs_grid.sel(time=slice(obs_start, END)))
print({var: f"{obs_grid[var].dropna('time', how='all').time.dt.date.values[[0, -1]]}" for var in obs_grid.data_vars})

### Lapse rates and zone series

The monthly lapse rate is the slope of the cells' mean temperature against cell elevation. It is fitted over all cells around the AOI and all observed years. In winter it is close to zero because cold air pools in the valleys (inversions).

In [ ]:
lapse = fit_lapse_rates(obs_grid, cells)
lapse.to_csv(CLIMATE_DIR / "lapse_rates.csv")
obs_grid_zones = grid_to_zones(obs_grid, weights, lapse)
save(obs_grid_zones, CLIMATE_DIR / "obs_grid_zones.nc")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
(lapse * 100).plot(ax=axes[0], marker="o")
axes[0].set_ylabel("K per 100 m")
axes[0].set_title("Monthly lapse rates (gridded observations)")
monthly_t = obs_grid_zones.tas.groupby("month").mean().to_pandas()
monthly_t.drop(columns="all").plot(ax=axes[1], cmap="viridis")
axes[1].set_ylabel("°C")
axes[1].set_title("Mean daily temperature per zone")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. CH2025 DAILY-GRIDDED scenarios

Each file holds one variable, model chain and period for all of Switzerland (~1.8 GB). It is downloaded, clipped to the AOI box (~10 MB) and deleted. The clipped files are shared by all AOIs whose box they cover. Files that fail (e.g. a dropped connection) are retried on the next run; the zone series are built from the files available.

In [ ]:
GRID_RE = re.compile(r"ogd-climate-scenarios-ch2025-grid_ch_([a-z]+)_(.+)_(ref91-20|gwl[\d.]+)\.nc$")
grid_dir = CLIMATE_SOURCE_DIR / "ch2025_grid"
grid_dir.mkdir(parents=True, exist_ok=True)


def grid_files():
    rows = []
    for name, asset in stac_get(f"{STAC_URL}{GRID_COLLECTION}/items/ch")["assets"].items():
        m = GRID_RE.match(name)
        if m and m.group(1) in CH2025_GRID_VARS and m.group(3) in CH2025_PERIODS \
                and (CH2025_GRID_MODELS is None or m.group(2) in CH2025_GRID_MODELS):
            rows.append({"var": m.group(1), "model": m.group(2), "period": m.group(3), "href": asset["href"],
                         "name": name, "path": grid_dir / f"{m.group(1)}_{m.group(2)}_{m.group(3)}.nc"})
    return pd.DataFrame(rows)


def fetch_grid(row):
    tmp = download(row.href, TMP_DIR / row.name)
    try:
        with xr.open_dataset(tmp) as ds:
            clipped = clip_grid(ds[[row.var]], row.box).load()
        part = row.path.with_name(row.path.name + ".part")
        save(clipped, part)
        part.replace(row.path)
    finally:
        tmp.unlink(missing_ok=True)


if DOWNLOAD_CH2025_GRID:
    files = grid_files()
    files["box"] = files.path.map(lambda p: fetch_bbox(p, bbox))   # None: cached file covers this AOI
    todo = files[files.box.notna()]
    print(f"{len(files)} files wanted, {len(todo)} to download (missing or not covering this AOI)")
    failed = []
    with ThreadPoolExecutor(GRID_WORKERS) as pool:
        futures = {pool.submit(fetch_grid, row): row for row in todo.itertuples()}
        for i, future in enumerate(as_completed(futures), 1):
            row = futures[future]
            try:
                future.result()
            except Exception as e:
                failed.append(row.name)
                print(f"  failed {row.name}: {e!r}")
            if i % 20 == 0 or i == len(futures):
                print(f"  {i}/{len(futures)} done ({pd.Timestamp.now():%H:%M})")
    if failed:
        print(f"{len(failed)} files failed: re-run this cell to retry them")

In [ ]:
def grid_zone_series():
    # Zone series of all clipped CH2025 grid files -> dims (period, model, time, zone)
    found = pd.DataFrame([dict(zip(["var", "model", "period"], p.stem.split("_")), path=p)
                          for p in sorted(grid_dir.glob("*.nc")) if fetch_bbox(p, bbox) is None])
    by_period = {}
    for period, pf in found.groupby("period", sort=False):
        by_model = {}
        for model, mf in pf.groupby("model"):
            ds = add_calendar(xr.merge([load_clipped(p) for p in mf.path], join="inner"))
            by_model[model] = grid_to_zones(ds, weights, lapse)
        by_period[period] = xr.concat(list(by_model.values()), pd.Index(list(by_model), name="model"), join="outer")
    order = [p for p in CH2025_PERIODS if p in by_period]
    return xr.concat([by_period[p] for p in order], pd.Index(order, name="period"), join="outer"), found


if any(grid_dir.glob("*.nc")):
    ch2025_grid_zones, found = grid_zone_series()
    save(ch2025_grid_zones, CLIMATE_DIR / "ch2025_grid_zones.nc")
    display(found.groupby(["period", "var"]).size().unstack().rename_axis(columns="files per variable"))
else:
    print("No CH2025 grid files yet")